In [7]:

# this script creates a pivot table from the matched combined statements

import pandas as pd
import os
import sys
import shutil
import zipfile
import tempfile
import xml.etree.ElementTree as ET
from datetime import datetime
from pathlib import Path
from xml.sax.saxutils import escape

karen_root = '/Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM group/Royalties/Statements/Karen'
output_root = '/Users/johannesnatterer/Developer/_output'

file = f'{karen_root}/All labels combined/Combined_statements_until_2026Q2_matched.csv'
analysisfile = f'{karen_root}/All labels combined/Combined_statements_until_2026Q2_analysis.xlsx'
outputfilename3 = f'{output_root}/Combined_statements_until_2026Q2_analysis.xlsx'

outputdirectory = os.path.dirname(outputfilename3)
os.makedirs(outputdirectory, exist_ok=True)
logfile_name = os.path.splitext(os.path.basename(outputfilename3))[0] + f'_run_log_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt'
log_path = os.path.join(outputdirectory, logfile_name)
_log_file = open(log_path, 'w', encoding='utf-8')
_original_stdout = sys.stdout.streams[0] if type(sys.stdout).__name__ == '_Tee' else sys.stdout

class _Tee:
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            s.write(data)
        self.flush()
    def flush(self):
        for s in self.streams:
            s.flush()
    def isatty(self):
        return False
    def __getattr__(self, name):
        return getattr(self.streams[0], name)

sys.stdout = _Tee(_original_stdout, _log_file)
pd.options.display.float_format = '{:,.2f}'.format
pd.set_option('display.max_columns', 12)
pd.set_option('display.width', 140)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.max_rows', 20)

def close_log():
    sys.stdout.flush()
    sys.stdout = _original_stdout
    if _log_file and not _log_file.closed:
        _log_file.close()
    print(f"Run log saved to: {log_path}")

def fmt_int(n):
    try:
        return f"{int(n):,}"
    except (TypeError, ValueError):
        return str(n)

def fmt_money(n):
    return f"{float(n):,.2f}"

def fmt_units(n):
    return f"{float(n):,.2f}"

def fmt_shape(df):
    return f"{fmt_int(df.shape[0])} rows × {len(df.columns)} columns"

def header(title):
    line = "=" * 72
    print(f"\n{line}\n  {title}\n{line}")

def subheader(title):
    print(f"\n--- {title} ---")

def print_totals(label, df, royalty_col='Royalty (CNY)', units_col='Units'):
    royalty = df[royalty_col].sum()
    units = df[units_col].sum()
    print(
        f"  {label:<24} Royalty (CNY): {fmt_money(royalty):>16}"
        f"    Units: {fmt_units(units):>20}    ({fmt_shape(df)})"
    )

def _col_letter(n):
    letters = ""
    while n:
        n, remainder = divmod(n - 1, 26)
        letters = chr(65 + remainder) + letters
    return letters

def _xml_text(value):
    text = "".join(ch for ch in str(value) if ch in "\t\n\r" or ord(ch) >= 32)
    return escape(text)

def _write_sheet_xml(df, xml_path):
    """Write a worksheet using inline strings so the rest of the xlsx can keep its sharedStrings.xml."""
    n_rows, n_cols = df.shape
    last_cell = f"{_col_letter(n_cols)}{n_rows + 1}"
    numeric_cols = [pd.api.types.is_numeric_dtype(df[col]) for col in df.columns]
    with open(xml_path, "w", encoding="utf-8") as fh:
        fh.write('<?xml version="1.0" encoding="UTF-8" standalone="yes"?>')
        fh.write('<worksheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main">')
        fh.write(f'<dimension ref="A1:{last_cell}"/>')
        fh.write('<sheetData>')
        fh.write('<row r="1">')
        for idx, col in enumerate(df.columns, start=1):
            fh.write(
                f'<c r="{_col_letter(idx)}1" t="inlineStr"><is><t>{_xml_text(col)}</t></is></c>'
            )
        fh.write("</row>")
        values = df.to_numpy()
        for row_idx, row in enumerate(values, start=2):
            fh.write(f'<row r="{row_idx}">')
            for col_idx, value in enumerate(row):
                if value is None or (isinstance(value, float) and pd.isna(value)) or pd.isna(value):
                    continue
                cell_ref = f"{_col_letter(col_idx + 1)}{row_idx}"
                if numeric_cols[col_idx]:
                    fh.write(f'<c r="{cell_ref}" t="n"><v>{value}</v></c>')
                else:
                    fh.write(
                        f'<c r="{cell_ref}" t="inlineStr"><is><t xml:space="preserve">{_xml_text(value)}</t></is></c>'
                    )
            fh.write("</row>")
        fh.write("</sheetData></worksheet>")

def _sheet_zip_path(xlsx_path, sheet_name):
    ns = {
        "m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
        "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
        "pr": "http://schemas.openxmlformats.org/package/2006/relationships",
    }
    with zipfile.ZipFile(xlsx_path) as zf:
        workbook = ET.fromstring(zf.read("xl/workbook.xml"))
        rels = ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))
    rel_id = None
    for sheet in workbook.findall("m:sheets/m:sheet", ns):
        if sheet.get("name") == sheet_name:
            rel_id = sheet.get("{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id")
            break
    if rel_id is None:
        return None
    target = None
    for rel in rels:
        if rel.get("Id") == rel_id:
            target = rel.get("Target")
            break
    if not target:
        return None
    target = target.lstrip("/")
    if target.startswith("xl/"):
        return target
    return "xl/" + target

def _file_on_disk(path):
    """False for OneDrive cloud-only placeholders (logical size but 0 blocks)."""
    try:
        st = os.stat(path)
    except OSError:
        return False
    return st.st_size > 0 and getattr(st, "st_blocks", 1) > 0


def _write_pivot_workbook(pivot_df, output_path, sheet_name="Data"):
    """Always produce a local xlsx with the pivot data."""
    if os.path.exists(output_path) and os.path.getsize(output_path) == 0:
        os.remove(output_path)
    print(f"  Writing Data sheet     : {fmt_int(len(pivot_df))} rows -> {output_path}")
    sys.stdout.flush()
    try:
        pivot_df.to_excel(output_path, sheet_name=sheet_name, index=False, engine="xlsxwriter")
    except ImportError:
        pivot_df.to_excel(output_path, sheet_name=sheet_name, index=False, engine="openpyxl")
    size = os.path.getsize(output_path)
    print(f"  Workbook saved         : {output_path}  ({size:,} bytes)")
    sys.stdout.flush()


def save_pivot_into_analysis(pivot_df, analysis_path, output_path, sheet_name="Data"):
    """Write the pivot locally. Only splice into the analysis workbook if it is already on disk."""
    _write_pivot_workbook(pivot_df, output_path, sheet_name=sheet_name)

    if not os.path.exists(analysis_path):
        print("  Analysis template      : not found (Data workbook only)")
        return
    if not _file_on_disk(analysis_path):
        print(
            "  Analysis template      : OneDrive cloud-only (not downloaded).\n"
            "    Other sheets/pivots were not copied. In Finder: right-click\n"
            "    Combined_statements_until_2026Q2_analysis.xlsx → Download Now\n"
            "    / Always Keep on This Device, then re-run this cell."
        )
        return

    print("  Analysis template is on disk — replacing Data sheet, keeping other tabs")
    sys.stdout.flush()
    tmp_template = output_path + ".template.xlsx"
    try:
        shutil.copyfile(analysis_path, tmp_template)
        sheet_part = _sheet_zip_path(tmp_template, sheet_name)
        if sheet_part is None:
            print(f"  Sheet '{sheet_name}' not in template — keeping Data workbook")
            return
        with tempfile.TemporaryDirectory() as tmpdir:
            xml_path = os.path.join(tmpdir, "sheet.xml")
            _write_sheet_xml(pivot_df, xml_path)
            sheet_bytes = Path(xml_path).read_bytes()
            tmp_out = output_path + ".tmp"
            with zipfile.ZipFile(tmp_template, "r") as zin, zipfile.ZipFile(tmp_out, "w") as zout:
                for item in zin.infolist():
                    if item.filename == sheet_part:
                        zi = zipfile.ZipInfo(item.filename, item.date_time)
                        zi.compress_type = zipfile.ZIP_DEFLATED
                        zout.writestr(zi, sheet_bytes)
                    else:
                        zout.writestr(item, zin.read(item.filename))
            os.replace(tmp_out, output_path)
        print(f"  Updated analysis file  : {output_path}")
    finally:
        if os.path.exists(tmp_template):
            os.remove(tmp_template)
    sys.stdout.flush()


header("Karen combined statements — pivot")
print(f"  Run started : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Input file     : {file}")
print(f"  Analysis file  : {analysisfile}")
print(f"  Output file    : {outputfilename3}")
print(f"  Run log        : {logfile_name}")

header("1. Required files")
if os.path.exists(file):
    print(f"  [OK]       {file}")
else:
    print(f"  [MISSING]  {file}")
    raise FileNotFoundError(f"Stopping: required file not found:\n- {file}")

if os.path.exists(analysisfile):
    print(f"  [OK]       {analysisfile}")
else:
    print(f"  [NEW]      {analysisfile}  (not found — a new workbook will be created)")

header("2. Load combined statements")
df = pd.read_csv(file, low_memory=False)
print(f"  Loaded: {fmt_shape(df)}")
print_totals("Input", df)

unnamed = [c for c in df.columns if str(c).startswith('Unnamed')]
if unnamed:
    print(f"  Dropping unnamed columns: {unnamed}")
    df = df.drop(columns=unnamed)

subheader("Columns")
for col in df.columns:
    type_counts = df[col].map(type).value_counts()
    types_str = ", ".join(f"{t.__name__}: {fmt_int(n)}" for t, n in type_counts.items())
    mixed = "  [mixed types]" if len(type_counts) > 1 else ""
    print(f"  {col:<22} {types_str}{mixed}")

# Filter one specific platform
# df = df[df['Platform'] == 'Spotify'].iloc[:, :]
# print(f"The filtered dataframe has {df.shape[0]} rows and {len(df.columns)} columns.")

 


  Karen combined statements — pivot
  Run started : 2026-08-21 12:01:42
  Input file     : /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM group/Royalties/Statements/Karen/All labels combined/Combined_statements_until_2026Q2_matched.csv
  Analysis file  : /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM group/Royalties/Statements/Karen/All labels combined/Combined_statements_until_2026Q2_analysis.xlsx
  Output file    : /Users/johannesnatterer/Developer/_output/Combined_statements_until_2026Q2_analysis.xlsx
  Run log        : Combined_statements_until_2026Q2_analysis_run_log_20260821_120142.txt

  1. Required files
  [OK]       /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM group/Royalties/Statements/Karen/All labels combined/Combined_statements_until_2026Q2_matched.csv
  [OK]       /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM group/Royalties/Statements/Karen/All labels combined/Comb

In [8]:
# Create Pivot table

header("3. Build pivot")

val1 = 'Units'
val2 = 'Royalty (CNY)'

val1crit = 'sum'
val2crit = 'sum'

x1 = 'Sales Quarter'
x2 = 'Source'
x3 = 'Platform'
x4 = 'Album_type'
x5 = 'Album'
x6 = 'Song'
x7 = 'Platform_original'
x8 = 'Statement Quarter'
x9 = 'Sales Month'
x10 = 'ISRC'
y1 = ''

sortby = val2

group_cols = [x2, x8, x1, x3,x5,x6, x10]
print(f"  Values : {val1} ({val1crit}), {val2} ({val2crit})")
print(f"  Index  : {', '.join(group_cols)}")

missing_cols = [c for c in group_cols + [val1, val2] if c not in df.columns]
if missing_cols:
    print(f"  ERROR: missing columns: {missing_cols}")
    print(f"  Available columns: {df.columns.tolist()}")
    raise KeyError(f"Pivot is missing required columns: {missing_cols}")

for col in group_cols:
    empty = df[col].isna().sum()
    if empty:
        print(f"  Filling {fmt_int(empty)} empty values in '{col}' with 'Blank'")
    df[col] = df[col].fillna("Blank")

pivot = pd.pivot_table(df, 
                        values=[val1, val2],
                        index=group_cols,
                        fill_value=0,
                        margins=False,
                        aggfunc="sum")

pivot = pivot.reset_index()
pivot[group_cols] = pivot[group_cols].ffill()

print(f"  Pivot shape: {fmt_shape(pivot)}")
print_totals("Input", df)
print_totals("Pivot", pivot)

subheader("Totals by source")
summary = (
    pivot.groupby("Source", sort=False)
    .agg(Rows=(val2, "size"), Royalty_CNY=(val2, "sum"), Units=(val1, "sum"))
)
print(f"  {'Source':<16} {'Rows':>12} {'Royalty (CNY)':>18} {'Units':>20}")
print(f"  {'-'*16} {'-'*12} {'-'*18} {'-'*20}")
for source, row in summary.iterrows():
    print(
        f"  {source:<16} {fmt_int(row['Rows']):>12} "
        f"{fmt_money(row['Royalty_CNY']):>18} {fmt_units(row['Units']):>20}"
    )
print(f"  {'-'*16} {'-'*12} {'-'*18} {'-'*20}")
print(
    f"  {'TOTAL':<16} {fmt_int(len(pivot)):>12} "
    f"{fmt_money(pivot[val2].sum()):>18} "
    f"{fmt_units(pivot[val1].sum()):>20}"
)

subheader("Sample (first 10 rows)")
print(pivot.head(10).to_string(index=False))

#pivot = pd.pivot_table(df, 
#                        values=[val1,val2],
#                        index=[x2,x8, x1,x3,x6,x10],
#                        #index=[x3,x7],
#                        #index=[x6,x7,x2], 
##                        columns= y1,
#                        fill_value='',
#                        margins= False,
#                        aggfunc={val1:val1crit, val2:val2crit})
#
#pivot = pivot.ffill(axis=1).bfill(axis=1)

header("4. Save output")
try:
    save_pivot_into_analysis(pivot, analysisfile, outputfilename3, sheet_name="Data")
    print(f"  Wrote pivot to 'Data' : {fmt_shape(pivot)}")
    print(f"  Saved                 : {outputfilename3}")
    print(f"\n  Run finished : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
finally:
    close_log()



  3. Build pivot
  Values : Units (sum), Royalty (CNY) (sum)
  Index  : Source, Statement Quarter, Sales Quarter, Platform, Album, Song, ISRC
  Pivot shape: 109,375 rows × 9 columns
  Input                    Royalty (CNY):    34,901,283.39    Units:    20,712,796,845.17    (3,670,885 rows × 12 columns)
  Pivot                    Royalty (CNY):    34,901,283.39    Units:    20,712,796,845.17    (109,375 rows × 9 columns)

--- Totals by source ---
  Source                   Rows      Royalty (CNY)                Units
  ---------------- ------------ ------------------ --------------------
  Allsaints                 130           4,887.15           855,011.00
  Believe                 5,720         535,298.11       126,540,216.00
  Douyin                    485          32,352.14        22,895,202.00
  Earth                  28,654       3,082,627.63       354,741,920.00
  Netease                 1,757       5,002,948.67     1,415,644,157.00
  Rock                   33,705       3,592,

KeyboardInterrupt: 